# 05 · analysis v2 (CPU): all seeds + training dynamics
Runtime: **CPU**. Run **after** notebook 02b (training) has finished **and** 03c has been re-run so the new seed-1/2 runs are tested on DMID.
Reads everything, retrains nothing, overwrites nothing; writes `results/p2_analysis_v2/` (v1 stays in `p2_analysis`).
New vs v1: run-level and seed-level SDs, per-seed table, CBIS per-image scores averaged over seeds before paired tests, and a training-dynamics check (train vs val Dice gap at the best epoch, learning-rate collapse) for every run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/quantum-placement'

In [ ]:
# 05 · analysis v2 (CPU only): all seeds + training-dynamics check. Reads finished CBIS runs + DMID test outputs, writes results/p2_analysis_v2/.
import os, glob, json, hashlib, re
import numpy as np, pandas as pd
from scipy.stats import wilcoxon

PROJECT = globals().get('PROJECT', '/content/drive/MyDrive/quantum-placement')
CBIS = f'{PROJECT}/results/p2_cbis_runs_e200'
DMID = f'{PROJECT}/results/p2_external_dmid_roi'
DMID_META = f'{PROJECT}/data/external/dmid_mass_256/roi'
OUT = f'{PROJECT}/results/p2_analysis_v2'      # v1 (seed 0 only) stays in p2_analysis
os.makedirs(OUT, exist_ok=True)
MODELS = ['unet', 'full', 'g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
SEVERABLE = ['g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
B = 2000
rng = np.random.default_rng(0)

def norm_cols(df):
    df.columns = [c.strip() for c in df.columns]
    return df

# ---------------------------------------------------------------- 1. runs table
rows = []
for d in sorted(glob.glob(f'{CBIS}/*_fold*_seed*')):
    f = f'{d}/summary.json'
    if not os.path.exists(f):
        continue
    s = json.load(open(f))
    r = {k: v for k, v in s.items() if not isinstance(v, (dict, list))}
    r['run'] = os.path.basename(d)
    for tag in ('grad_audit_init', 'grad_audit_best'):
        g = s.get(tag)
        if isinstance(g, dict) and 'per_weight' in g:
            w = np.abs(np.array(g['per_weight'], float))          # (layers, qubits, 3)
            inert = np.zeros(w.shape, bool); inert[-1, :, 2] = True  # final-layer omega: no effect on <Z>
            eff = w[~inert]
            r[f'{tag}_max'] = eff.max()
            r[f'{tag}_median'] = np.median(eff)
            r[f'{tag}_min'] = eff.min()
            r[f'{tag}_n_above_1pct_max'] = int((eff >= 0.01 * eff.max()).sum())   # relative threshold
            r[f'{tag}_inert_max'] = w[inert].max()
    rows.append(r)
runs = pd.DataFrame(rows)
print('CBIS runs found:', len(runs), '| per model x seed:', runs.groupby(['model', 'seed']).size().to_dict())
dtab = f'{PROJECT}/results/p2_external_dmid_roi_table.csv'
if os.path.exists(dtab):
    dt = pd.read_csv(dtab)
    keep = [c for c in dt.columns if c.startswith('dmid_') or c in ('run', 'weights_ok', 'n_dmid')]
    runs = runs.merge(dt[keep], on='run', how='left')
runs.to_csv(f'{OUT}/runs.csv', index=False)

# ---------------------------------------------------------------- 2. per-image tables
cb = []
for d in sorted(glob.glob(f'{CBIS}/*_fold*_seed*')):
    f = f'{d}/per_image.csv'
    if os.path.exists(f):
        t = norm_cols(pd.read_csv(f)); run = os.path.basename(d)
        t['model'] = run.split('_fold')[0]; t['fold'] = int(re.search(r'fold(\d+)', run).group(1))
        t['seed'] = int(re.search(r'seed(\d+)', run).group(1))
        cb.append(t)
cb = pd.concat(cb, ignore_index=True)
cb.to_csv(f'{OUT}/cbis_per_image_all.csv', index=False)

# DMID metadata + duplicate detection by identical (image, mask)
meta = pd.read_csv(f'{DMID_META}/metadata.csv')
def mhash(rid):
    p = f'{DMID_META}/masks/{rid}.png'
    return hashlib.md5(open(p, 'rb').read()).hexdigest() if os.path.exists(p) else rid
meta['mask_md5'] = meta.roi_id.map(mhash)
meta['dup_key'] = meta.img.astype(str) + '_' + meta.mask_md5
meta['keep'] = ~meta.duplicated('dup_key')
print('DMID crops:', len(meta), '| unique (image, mask):', int(meta.keep.sum()), '| images:', meta.img.nunique())
meta[['roi_id', 'img', 'dup_key', 'keep']].to_csv(f'{OUT}/dmid_dedup.csv', index=False)

dm = []
for d in sorted(glob.glob(f'{DMID}/*_fold*_seed*')):
    f = f'{d}/per_image.csv'
    if os.path.exists(f):
        t = norm_cols(pd.read_csv(f)); run = os.path.basename(d)
        if 'roi_id' not in t.columns:
            t = t.rename(columns={t.columns[0]: 'roi_id'})
        t['model'] = run.split('_fold')[0]; t['fold'] = int(re.search(r'fold(\d+)', run).group(1))
        t['seed'] = int(re.search(r'seed(\d+)', run).group(1))
        dm.append(t)
dm = pd.concat(dm, ignore_index=True)
print('DMID per-image columns:', list(dm.columns))
dm = dm.drop(columns=[c for c in ('img', 'keep') if c in dm.columns]).merge(meta[['roi_id', 'img', 'keep']], on='roi_id', how='left')
print('DMID rows without metadata match:', int(dm.img.isna().sum()))
dm.to_csv(f'{OUT}/dmid_per_image_all.csv', index=False)

# ---------------------------------------------------------------- 3. per-model summary (run level and seed level)
def run_stats(df, col):
    r = df.groupby(['model', 'fold', 'seed'])[col].mean()                       # one value per run
    by_seed = r.groupby(['model', 'seed']).mean()                                 # 5-fold mean per seed
    return r.groupby('model').mean(), r.groupby('model').std(), by_seed.groupby('model').std()
summ = pd.DataFrame(index=MODELS)
summ['n_runs'] = runs.groupby('model').size(); summ['n_seeds'] = runs.groupby('model').seed.nunique()
for name, df, cols in (('cbis', cb, ['dice', 'iou', 'dice_const', 'dice_swap']),
                       ('dmid_dedup', dm[dm.keep == True], ['dice', 'iou', 'dice_const', 'dice_swap'])):
    for c in cols:
        if c in df.columns:
            m, s_run, s_seed = run_stats(df, c)
            summ[f'{name}_{c}_mean'] = m; summ[f'{name}_{c}_sd_runs'] = s_run
            if c == 'dice': summ[f'{name}_{c}_sd_seeds'] = s_seed
    if 'hd95' in df.columns:
        summ[f'{name}_hd95_median'] = df.groupby('model').hd95.median()
for c in ('epochs_run', 'best_epoch', 'train_seconds', 'total_params', 'trainable_params'):
    if c in runs.columns:
        summ[c] = runs.groupby('model')[c].mean()
summ = summ.round(4)
summ.to_csv(f'{OUT}/model_summary.csv')
print(summ.to_string())
per_seed = pd.concat({'cbis_dice': cb.groupby(['model', 'seed', 'fold']).dice.mean().groupby(['model', 'seed']).mean(),
                      'dmid_dice': dm[dm.keep == True].groupby(['model', 'seed', 'fold']).dice.mean().groupby(['model', 'seed']).mean()}, axis=1).round(4)
per_seed.to_csv(f'{OUT}/per_seed.csv'); print(per_seed.to_string())

# ---------------------------------------------------------------- 4. paired comparisons with cluster bootstrap
def paired(df, a, b, col, cluster):
    x = df[df.model == a].set_index('key'); y = df[df.model == b].set_index('key')
    k = x.index.intersection(y.index)
    diff = (x.loc[k, col] - y.loc[k, col]).rename('d').to_frame()
    diff['cl'] = x.loc[k, cluster].values
    per = diff.groupby('cl').d.mean()                       # cluster (patient / image) means
    cls = per.index.to_numpy(); n = len(cls)
    cl_sum = diff.groupby('cl').d.sum(); cl_n = diff.groupby('cl').size()
    boots = []
    for _ in range(B):
        s = rng.choice(cls, n, replace=True)
        boots.append(cl_sum.loc[s].sum() / cl_n.loc[s].sum())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = wilcoxon(per.values).pvalue if (per != 0).any() else 1.0
    return dict(A=a, B=b, metric=col, n_images=len(k), n_clusters=n, mean_diff=diff.d.mean(), ci_low=lo, ci_high=hi, p_wilcoxon_cluster=p)

CONTRASTS = [('g_q', 'g_cl'), ('g_q', 'g_id'), ('g_cl', 'g_id'), ('p_q', 'p_cl'), ('p_q', 'p_id'), ('p_cl', 'p_id'),
             ('p_q', 'g_q'), ('p_cl', 'g_cl'), ('p_id', 'g_id'), ('full', 'g_id'), ('full', 'p_id'), ('unet', 'full')]
res = []
cbk = cb.groupby(['model', 'roi_id', 'patient'], as_index=False)[[c for c in ('dice', 'dice_const', 'dice_swap') if c in cb.columns]].mean()  # mean over seeds
cbk['key'] = cbk.roi_id
for a, b in CONTRASTS:
    res.append(dict(dataset='CBIS (pooled test folds, mean over seeds)', **paired(cbk, a, b, 'dice', 'patient')))
# DMID: average each crop's score over all fold x seed models, de-duplicated crops, cluster = DMID image
dmm = dm[dm.keep == True].groupby(['model', 'roi_id', 'img'], as_index=False)[[c for c in ('dice', 'dice_const', 'dice_swap') if c in dm.columns]].mean()
dmm['key'] = dmm.roi_id
for a, b in CONTRASTS:
    res.append(dict(dataset='DMID (dedup, mean of all fold x seed models)', **paired(dmm, a, b, 'dice', 'img')))
res = pd.DataFrame(res)
for ds in res.dataset.unique():                              # Holm correction within each dataset
    m = res.dataset == ds; p = res.loc[m, 'p_wilcoxon_cluster'].values; o = np.argsort(p); k = len(p)
    adj = np.empty(k); run_max = 0
    for i, j in enumerate(o):
        run_max = max(run_max, (k - i) * p[j]); adj[j] = min(1, run_max)
    res.loc[m, 'p_holm'] = adj
res.round(5).to_csv(f'{OUT}/paired_comparisons.csv', index=False)
print(res.round(4).to_string())

# ---------------------------------------------------------------- 5. severing effect (normal - severed)
sev = []
for name, df, cl in (('CBIS', cbk, 'patient'), ('DMID', dmm, 'img')):
    for mname in SEVERABLE:
        t = df[df.model == mname]
        for mode in ('const', 'swap'):
            c = f'dice_{mode}'
            if c not in t.columns:
                continue
            d = (t.dice - t[c]).rename('d').to_frame(); d['cl'] = t[cl].values
            cl_sum = d.groupby('cl').d.sum(); cl_n = d.groupby('cl').size(); cls = cl_sum.index.to_numpy()
            boots = [cl_sum.loc[s].sum() / cl_n.loc[s].sum() for s in (rng.choice(cls, len(cls)) for _ in range(B))]
            per = d.groupby('cl').d.mean()
            sev.append(dict(dataset=name, model=mname, mode=mode, dice=t.dice.mean(), dice_severed=t[c].mean(),
                            drop=d.d.mean(), ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5),
                            p_wilcoxon=wilcoxon(per.values).pvalue, frac_images_worse=(d.d > 0).mean()))
sev = pd.DataFrame(sev)
sev.round(5).to_csv(f'{OUT}/severing.csv', index=False)
print(sev.round(4).to_string())

# ---------------------------------------------------------------- 6. gradient audit
ga = runs[runs.model.isin(['g_q', 'p_q'])][['run', 'model', 'fold', 'seed'] + [c for c in runs.columns if c.startswith('grad_audit')]]
ga.round(8).to_csv(f'{OUT}/gradient_audit.csv', index=False)
print(ga.to_string())

# ---------------------------------------------------------------- 6b. training dynamics (over/under-fitting check)
hrows = []
for d in sorted(glob.glob(f'{CBIS}/*_fold*_seed*')):
    f = f'{d}/history.csv'
    if not os.path.exists(f):
        continue
    h = norm_cols(pd.read_csv(f)); run = os.path.basename(d)
    b = int(h.val_loss.idxmin())
    lr_col = 'learning_rate' if 'learning_rate' in h.columns else ('lr' if 'lr' in h.columns else None)
    low = h.index[h[lr_col] <= 1e-6] if lr_col else []
    hrows.append(dict(run=run, model=run.split('_fold')[0], fold=int(re.search(r'fold(\d+)', run).group(1)),
                      seed=int(re.search(r'seed(\d+)', run).group(1)), epochs=len(h), best_epoch=b + 1,
                      train_dice_best=h.soft_dice[b], val_dice_best=h.val_soft_dice[b],
                      gap_dice=h.soft_dice[b] - h.val_soft_dice[b], train_loss_best=h.loss[b], val_loss_best=h.val_loss[b],
                      train_dice_max=h.soft_dice.max(), val_loss_max_first10=h.val_loss[:10].max(),
                      lr_min_epoch=(int(low[0]) + 1 if len(low) else np.nan), final_lr=(h[lr_col].iloc[-1] if lr_col else np.nan)))
dyn = pd.DataFrame(hrows)
dyn.round(5).to_csv(f'{OUT}/training_dynamics_runs.csv', index=False)
dyn_s = dyn.groupby('model').agg(n=('run', 'size'), train_dice_best=('train_dice_best', 'mean'), val_dice_best=('val_dice_best', 'mean'),
                                 gap_dice=('gap_dice', 'mean'), gap_dice_sd=('gap_dice', 'std'), train_dice_max=('train_dice_max', 'mean'),
                                 val_loss_max_first10=('val_loss_max_first10', 'median'), lr_min_epoch=('lr_min_epoch', 'median'),
                                 frac_runs_lr_collapsed=('lr_min_epoch', lambda x: x.notna().mean())).reindex(MODELS).round(4)
dyn_s.to_csv(f'{OUT}/training_dynamics_summary.csv'); print(dyn_s.to_string())

# ---------------------------------------------------------------- 7. one Excel file + figure
with pd.ExcelWriter(f'{OUT}/P2_results.xlsx') as xw:
    summ.to_excel(xw, sheet_name='model_summary')
    per_seed.to_excel(xw, sheet_name='per_seed')
    dyn_s.to_excel(xw, sheet_name='training_dynamics')
    dyn.round(5).to_excel(xw, sheet_name='training_dynamics_runs', index=False)
    res.round(5).to_excel(xw, sheet_name='paired_comparisons', index=False)
    sev.round(5).to_excel(xw, sheet_name='severing', index=False)
    ga.round(8).to_excel(xw, sheet_name='gradient_audit', index=False)
    runs.round(5).to_excel(xw, sheet_name='runs', index=False)
    meta[['roi_id', 'img', 'keep']].to_excel(xw, sheet_name='dmid_dedup', index=False)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for a, (name, df) in zip(ax, (('CBIS-DDSM (internal, one dot per run)', cb), ('DMID (external, dedup)', dm[dm.keep == True]))):
    f = df.groupby(['model', 'fold', 'seed']).dice.mean().reset_index()
    for i, m in enumerate(MODELS):
        v = f[f.model == m].dice
        a.scatter([i] * len(v), v, s=18, alpha=.7); a.plot([i - .25, i + .25], [v.mean()] * 2, 'k-')
    a.set_xticks(range(len(MODELS))); a.set_xticklabels(MODELS); a.set_title(name); a.grid(axis='y', alpha=.3)
ax[0].set_ylabel('per-image Dice (run mean)')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_dice_by_model.png', dpi=150)
print('saved to', OUT, os.listdir(OUT))
